# 24 -- position-prior probe: is the model shortcutting quadrant questions by class?

Not a new training rung. This is a **zero-GPU, read-only diagnostic** on predictions the
pod already has, motivated by ["Your other Left! Vision-Language Models Fail to Identify
Relative Positions in Medical Images"](https://arxiv.org/abs/2508.00549) (MICCAI 2025).
That paper's finding: VLMs answer medical position questions from a memorised **prior**
(typical anatomy) rather than reading the image -- exposed by testing accuracy on cases
where the true position CONTRADICTS the prior. Their own fix attempts (visual markers,
prompt variants) both failed; they never tested training-time intervention.

**The test here:** each foreign-object class has a measured "typical" quadrant in FRAME's
own train data (e.g. Needle is top/left or bottom/left ~74% of the time, and almost never
bottom/right). If the model is shortcutting on "class -> usual position" instead of
reading the frame, accuracy should collapse specifically on the ATYPICAL cases. If it
holds up, the model is genuinely reading position and the shortcut theory does not apply
here.

Run against **two** checkpoints so this also reads on the flip experiment itself: rung 21
arm A (`21_lr_1e4_v1`, unaffected by the flip augmentation) and `24_flip_p25_v1`. Even
though the flip arm's headline delta was not significant, it may still have moved the
typical/atypical GAP specifically -- a real mechanistic read the aggregate number is too
noisy to show.

**Pre-registered read:** report the atypical-minus-typical accuracy gap and its
video-clustered CI, for both arms, at the SAME epoch. A gap that excludes zero (negative)
supports the shortcut theory. Whether the flip arm's gap is measurably smaller than arm
A's gap is the mechanistic question this experiment was actually trying to answer.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, sys
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "24-geometric-aug":
    EXP = REPO / "experiments" / "24-geometric-aug"
for p in (REPO / "src", EXP / "_models",
          REPO / "experiments" / "21-recipe-sweep" / "_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

import flip_audit
import position_prior_probe as ppp
print("repo:", REPO)


In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects BELOW this cell) --------
EPOCH = 3                       # which epoch's results.csv to compare -- both arms, matched
RUN_ARM_A = "21_lr_1e4_v1"      # rung 21 arm A -- the current best checkpoint, unaffected by flip
RUN_FLIP = "24_flip_p25_v1"     # this experiment's p=0.25 arm
DATA_ROOT = "/workspace/orena-data"

# Explicit, INDEPENDENT of cwd-derived REPO below: on this pod, rung 21's control
# results live under the shared clone, but this experiment's own runs/ landed under a
# separate checkout -- a single cwd-walked REPO cannot reach both. Check with
# `find /workspace -maxdepth 1 -iname "repo*"` if these ever need to change.
ARM_A_REPO = "/workspace/repo"
RUNG24_REPO = "/workspace/repo_yyy"


In [ ]:
# --- derived ---------------------------------------------------------------------
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found -- pull the QA parquets"

ARM_A_RESULTS = (Path(ARM_A_REPO) / "experiments" / "21-recipe-sweep" / "runs" / RUN_ARM_A
                 / f"ep{EPOCH}_full" / "results.csv")
FLIP_RESULTS = (Path(RUNG24_REPO) / "experiments" / "24-geometric-aug" / "runs" / RUN_FLIP
                / f"ep{EPOCH}_full" / "results.csv")
for label, p in (("arm A", ARM_A_RESULTS), ("flip p25", FLIP_RESULTS)):
    assert p.exists(), f"{label} results.csv missing at {p} -- has epoch {EPOCH} been scored?"
print("arm A results:", ARM_A_RESULTS)
print("flip results: ", FLIP_RESULTS)


In [ ]:
# --- 1. the typical-quadrant map, from TRAIN only (never circular with test) ----
import glob as _glob

def _load_split(split):
    parts = []
    for f in sorted(_glob.glob(str(DATA_ROOT / "*" / "data" / "frame" / f"{split}.parquet"))):
        ds = Path(f).parents[2].name
        df = pd.read_parquet(f)
        df["dataset"] = ds
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

train_df = _load_split("train")
train_audit = flip_audit.audit_dataframe(train_df, split="train")
train_pairs = ppp.class_quadrant_rows(train_audit)
print(f"train (class, quadrant) observations: {len(train_pairs)}")

MODAL = ppp.typical_quadrant_map(train_pairs)
print(f"\nclasses with >= {ppp.MIN_CLASS_N} train observations "
      f"({len(MODAL)}/{train_pairs['class'].nunique()}):")
counts = train_pairs.groupby("class").size()
for cls, quad in sorted(MODAL.items(), key=lambda kv: -counts[kv[0]]):
    share = (train_pairs[train_pairs["class"] == cls]["quadrant"] == quad).mean()
    print(f"  {cls:16s} typical={quad:14s} n={counts[cls]:4d}  share={share:.1%}")
dropped = set(train_pairs["class"].unique()) - set(MODAL)
if dropped:
    print(f"\ndropped for low n (< {ppp.MIN_CLASS_N}): {sorted(dropped)}")


In [ ]:
# --- 2. label the TEST quadrant rows typical / atypical -------------------------
test_df = _load_split("test")
test_audit = flip_audit.audit_dataframe(test_df, split="test")
test_pairs = ppp.class_quadrant_rows(test_audit)
labelled = ppp.label_typicality(test_pairs, MODAL)

print(f"test (class, quadrant) observations: {len(test_pairs)}")
print(f"labelled (class has a train-derived typical quadrant): {len(labelled)}")
print(f"  typical:  {int(labelled['is_typical'].sum())}")
print(f"  atypical: {int((~labelled['is_typical']).sum())}")
print()
print(labelled.groupby(["rule", "is_typical"]).size().unstack(fill_value=0))


In [ ]:
# --- 3. score both arms, ALL rules combined --------------------------------------
result_arm_a = ppp.score_against_run(labelled, ARM_A_RESULTS)
result_flip = ppp.score_against_run(labelled, FLIP_RESULTS)

def _print_result(name, r):
    t, a = r["typical"], r["atypical"]
    print(f"--- {name} (all 3 rules combined) ---")
    print(f"  typical:  n={t['n']:4d} ({t['n_videos']} videos)  acc={t['acc']:.4f}")
    print(f"  atypical: n={a['n']:4d} ({a['n_videos']} videos)  acc={a['acc']:.4f}")
    print(f"  gap (atypical - typical): {r['gap_atypical_minus_typical']:+.4f}  "
          f"CI=[{r['gap_ci_low']:+.4f}, {r['gap_ci_high']:+.4f}]  "
          f"excludes_0={not (r['gap_ci_low'] <= 0 <= r['gap_ci_high'])}")
    print()

_print_result(f"rung 21 arm A (ep{EPOCH})", result_arm_a)
_print_result(f"flip p=0.25 (ep{EPOCH})", result_flip)


In [ ]:
# --- 4. the cleaner cut: single-item rules only (excludes all_object_positions) --
# `all_object_positions` grades a whole LIST per qID, so `correctness` on any one named
# object is slightly overstated by a wrong sibling item elsewhere in the same list.
# fixed_quadrant_class + object_center_quadrant are exactly one class per qID.
single_item = labelled[labelled.rule != "all_object_positions"]
print(f"single-item rows: {len(single_item)} "
      f"(typical={int(single_item['is_typical'].sum())}, "
      f"atypical={int((~single_item['is_typical']).sum())})")
print()

result_arm_a_si = ppp.score_against_run(single_item, ARM_A_RESULTS)
result_flip_si = ppp.score_against_run(single_item, FLIP_RESULTS)
_print_result(f"rung 21 arm A (ep{EPOCH}), single-item rules only", result_arm_a_si)
_print_result(f"flip p=0.25 (ep{EPOCH}), single-item rules only", result_flip_si)


In [ ]:
# --- 5. the pre-registered read --------------------------------------------------
def _verdict(r):
    lo, hi = r["gap_ci_low"], r["gap_ci_high"]
    if lo != lo or hi != hi:  # NaN
        return "not computed"
    if hi < 0:
        return "SHORTCUT SUPPORTED -- atypical significantly worse than typical"
    if lo > 0:
        return "atypical significantly BETTER than typical (unexpected)"
    return "no significant gap -- shortcut theory not supported here"

print("PRE-REGISTERED READ (single-item rules, the cleaner cut)")
print(f"  arm A   : {_verdict(result_arm_a_si)}")
print(f"  flip p25: {_verdict(result_flip_si)}")

d_gap = result_flip_si["gap_atypical_minus_typical"] - result_arm_a_si["gap_atypical_minus_typical"]
print(f"\n  flip vs arm A, change in the gap itself: {d_gap:+.4f} "
      "(closer to 0 = flip narrowed the shortcut; more negative = flip widened it)")
print("\n  Read this ALONGSIDE 24b's own headline verdict -- this probe explains a")
print("  MECHANISM, it does not override the pre-registered proxy/margin_OOD result.")


In [ ]:
# --- persist (small, always -- this is diagnostic, not a scored training arm) ---
import pandas as pd
rows = []
for label, res, scope in (
    ("arm_A", result_arm_a, "all_rules"), ("flip_p25", result_flip, "all_rules"),
    ("arm_A", result_arm_a_si, "single_item"), ("flip_p25", result_flip_si, "single_item"),
):
    rows.append({
        "run": label, "epoch": EPOCH, "scope": scope,
        "n_typical": res["typical"]["n"], "acc_typical": res["typical"]["acc"],
        "n_atypical": res["atypical"]["n"], "acc_atypical": res["atypical"]["acc"],
        "gap": res["gap_atypical_minus_typical"],
        "gap_ci_low": res["gap_ci_low"], "gap_ci_high": res["gap_ci_high"],
    })
out_df = pd.DataFrame(rows)
out_path = EXP / f"RESULTS_position_prior_probe_ep{EPOCH}.csv"
out_df.to_csv(out_path, index=False)
print("wrote", out_path)
out_df


## How to read this

- A **significant negative gap** (atypical worse, CI excludes 0) supports the "Your other
  Left!" shortcut theory for FRAME's quadrant questions -- the model leans on
  class-typical position instead of reading the frame.
- **No significant gap** means this specific failure mode is not what's limiting FRAME's
  quadrant accuracy, at least not measurably at this sample size -- worth knowing before
  investing in any fix aimed at it.
- The arm-A-vs-flip comparison is the mechanistic complement to 24b's own headline: even a
  non-significant proxy delta could hide a real (or absent) shift in this specific gap.
- Small-sample caveat carries over from the class-position analysis: `Gallstone` and any
  other sub-`MIN_CLASS_N` class are excluded from `MODAL`, and n per cell here is smaller
  than the parent 871-row transformable set -- read the CI, not just the point estimate.
